# Processed Datasets Overview

This notebook inspects all `*_processed` tables and reports:
1. Data timeframe details
2. Available columns
3. Missing data information

In [1]:
from pathlib import Path
import re
import sys

import pandas as pd
from sqlalchemy import text

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.config.database import engine

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

In [2]:
# Discover all processed tables in public schema.
tables_query = text(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_type = 'BASE TABLE'
      AND table_name LIKE '%\_processed' ESCAPE '\\'
    ORDER BY table_name
    """
)

processed_tables = pd.read_sql(tables_query, engine)["table_name"].tolist()
print(f"Processed tables found: {len(processed_tables)}")
processed_tables

Processed tables found: 6


['btcusdt_15m_processed',
 'btcusdt_1d_processed',
 'btcusdt_1h_processed',
 'btcusdt_1mo_processed',
 'btcusdt_1w_processed',
 'btcusdt_4h_processed']

## 1) Data Timeframe Detail

In [3]:
name_pattern = re.compile(r"^(?P<symbol>.+?)_(?P<interval>[^_]+)_processed$")

def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'

def has_column(table_name: str, column_name: str) -> bool:
    q = text(
        """
        SELECT COUNT(*)
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name = :table_name
          AND column_name = :column_name
        """
    )
    count = pd.read_sql(q, engine, params={"table_name": table_name, "column_name": column_name}).iloc[0, 0]
    return bool(count)

timeframe_rows = []
for table_name in processed_tables:
    match = name_pattern.match(table_name)
    symbol = match.group("symbol").upper() if match else "UNKNOWN"
    interval = match.group("interval") if match else "unknown"

    has_open_time_ms = has_column(table_name, "open_time_ms")
    has_open_time = has_column(table_name, "open_time")

    if has_open_time_ms:
        q = text(f"SELECT COUNT(*) AS row_count, MIN(open_time_ms) AS min_open_time_ms, MAX(open_time_ms) AS max_open_time_ms FROM {quote_ident(table_name)}")
        row = pd.read_sql(q, engine).iloc[0]
        start_utc = pd.to_datetime(row["min_open_time_ms"], unit="ms", utc=True) if pd.notna(row["min_open_time_ms"]) else pd.NaT
        end_utc = pd.to_datetime(row["max_open_time_ms"], unit="ms", utc=True) if pd.notna(row["max_open_time_ms"]) else pd.NaT
    elif has_open_time:
        q = text(f"SELECT COUNT(*) AS row_count, MIN(open_time) AS min_open_time, MAX(open_time) AS max_open_time FROM {quote_ident(table_name)}")
        row = pd.read_sql(q, engine).iloc[0]
        start_utc = pd.to_datetime(row["min_open_time"], utc=True) if pd.notna(row["min_open_time"]) else pd.NaT
        end_utc = pd.to_datetime(row["max_open_time"], utc=True) if pd.notna(row["max_open_time"]) else pd.NaT
    else:
        q = text(f"SELECT COUNT(*) AS row_count FROM {quote_ident(table_name)}")
        row = pd.read_sql(q, engine).iloc[0]
        start_utc = pd.NaT
        end_utc = pd.NaT

    span_days = (end_utc - start_utc).total_seconds() / 86400 if pd.notna(start_utc) and pd.notna(end_utc) else None

    timeframe_rows.append({
        "table_name": table_name,
        "symbol": symbol,
        "interval": interval,
        "row_count": int(row["row_count"]),
        "start_utc": start_utc,
        "end_utc": end_utc,
        "span_days": round(span_days, 2) if span_days is not None else None,
        "time_column": "open_time_ms" if has_open_time_ms else ("open_time" if has_open_time else "not_found"),
    })

timeframe_df = pd.DataFrame(timeframe_rows).sort_values(["symbol", "interval"]).reset_index(drop=True)
timeframe_df

,table_name,symbol,interval,row_count,start_utc,end_utc,span_days,time_column
0,btcusdt_15m_processed,BTCUSDT,15m,303071,2017-08-17 04:00:00+00:00,2026-04-09 03:30:00+00:00,3156.98,open_time_ms
1,btcusdt_1d_processed,BTCUSDT,1d,3158,2017-08-17 00:00:00+00:00,2026-04-09 00:00:00+00:00,3157.00,open_time_ms
2,btcusdt_1h_processed,BTCUSDT,1h,75768,2017-08-17 04:00:00+00:00,2026-04-09 03:00:00+00:00,3156.96,open_time_ms
3,btcusdt_1mo_processed,BTCUSDT,1mo,105,2017-08-01 00:00:00+00:00,2026-04-01 00:00:00+00:00,3165.00,open_time_ms
4,btcusdt_1w_processed,BTCUSDT,1w,452,2017-08-14 00:00:00+00:00,2026-04-06 00:00:00+00:00,3157.00,open_time_ms
5,btcusdt_4h_processed,BTCUSDT,4h,18942,2017-08-17 04:00:00+00:00,2026-04-09 00:00:00+00:00,3156.83,open_time_ms


## 2) Columns There

In [4]:
columns_query = text(
    """
    SELECT
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = ANY(:tables)
    ORDER BY table_name, ordinal_position
    """
)

if processed_tables:
    columns_df = pd.read_sql(columns_query, engine, params={"tables": processed_tables})
else:
    columns_df = pd.DataFrame(columns=["table_name", "ordinal_position", "column_name", "data_type", "is_nullable"])

print(f"Total table-column rows: {len(columns_df)}")
print("\nFull table-column list (all rows):")
print(columns_df.to_string(index=False))

print("\nGrouped column list per table (fully expanded):")
grouped_columns_df = (
    columns_df.groupby("table_name", dropna=False)["column_name"]
    .apply(list)
    .reset_index(name="columns")
)

for _, row in grouped_columns_df.iterrows():
    print(f"\n{row['table_name']} ({len(row['columns'])} columns)")
    for i, col in enumerate(row["columns"], start=1):
        print(f"  {i:02d}. {col}")

grouped_columns_df

Total table-column rows: 342

Full table-column list (all rows):
           table_name  ordinal_position                  column_name                   data_type is_nullable
btcusdt_15m_processed                 1                           id                      bigint          NO
btcusdt_15m_processed                 2                    open_time timestamp without time zone          NO
btcusdt_15m_processed                 3                         open            double precision         YES
btcusdt_15m_processed                 4                         high            double precision         YES
btcusdt_15m_processed                 5                          low            double precision         YES
btcusdt_15m_processed                 6                        close            double precision         YES
btcusdt_15m_processed                 7                       volume            double precision         YES
btcusdt_15m_processed                 8                   close

,table_name,columns
0,btcusdt_15m_processed,"[id, open_time, open, high, low, close, volume..."
1,btcusdt_1d_processed,"[id, open_time, open, high, low, close, volume..."
2,btcusdt_1h_processed,"[id, open_time, open, high, low, close, volume..."
3,btcusdt_1mo_processed,"[id, open_time, open, high, low, close, volume..."
4,btcusdt_1w_processed,"[id, open_time, open, high, low, close, volume..."
5,btcusdt_4h_processed,"[id, open_time, open, high, low, close, volume..."


In [5]:
# Unique column list across all processed tables, plus schema consistency check.
unique_columns_df = (
    columns_df[["column_name"]]
    .drop_duplicates()
    .sort_values("column_name")
    .reset_index(drop=True)
)

columns_per_table = (
    columns_df.groupby("table_name")["column_name"]
    .apply(lambda s: tuple(sorted(set(s))))
)
all_tables_same_columns = columns_per_table.nunique() == 1

print(f"Total unique columns across processed tables: {len(unique_columns_df)}")
print(f"All tables share same columns: {all_tables_same_columns}")

display(unique_columns_df)

Total unique columns across processed tables: 57
All tables share same columns: True


,column_name
0,ad_line
1,adx_14
2,atr_14
3,bollinger_lower_20
4,bollinger_mid_20
5,bollinger_upper_20
6,candle_pattern_1
7,candle_pattern_2
8,candle_pattern_3
9,candle_pattern_complex


In [6]:
# Compact per-table column summary.
column_summary_df = (
    columns_df.groupby("table_name", dropna=False)
    .agg(
        total_columns=("column_name", "count"),
        nullable_columns=("is_nullable", lambda s: int((s == "YES").sum())),
    )
    .reset_index()
    .sort_values("table_name")
)
column_summary_df

,table_name,total_columns,nullable_columns
0,btcusdt_15m_processed,57,44
1,btcusdt_1d_processed,57,44
2,btcusdt_1h_processed,57,44
3,btcusdt_1mo_processed,57,44
4,btcusdt_1w_processed,57,44
5,btcusdt_4h_processed,57,44


## 3) Missing Data Info

In [7]:
missing_rows = []

for table_name in processed_tables:
    table_cols = columns_df.loc[columns_df["table_name"] == table_name, "column_name"].tolist()
    if not table_cols:
        continue

    null_exprs = [
        f"SUM(CASE WHEN {quote_ident(col)} IS NULL THEN 1 ELSE 0 END) AS {quote_ident(col)}"
        for col in table_cols
    ]
    nulls_sql = f"SELECT {', '.join(null_exprs)} FROM {quote_ident(table_name)}"
    null_counts = pd.read_sql(text(nulls_sql), engine).iloc[0].to_dict()

    row_count = int(timeframe_df.loc[timeframe_df["table_name"] == table_name, "row_count"].iloc[0])

    for col in table_cols:
        null_count = int(null_counts.get(col, 0) or 0)
        missing_pct = (null_count / row_count * 100.0) if row_count > 0 else None
        missing_rows.append({
            "table_name": table_name,
            "column_name": col,
            "row_count": row_count,
            "null_count": null_count,
            "missing_pct": round(missing_pct, 4) if missing_pct is not None else None,
        })

missing_df = pd.DataFrame(missing_rows)
missing_df = missing_df.sort_values(["table_name", "missing_pct", "null_count"], ascending=[True, False, False]).reset_index(drop=True)
missing_df

,table_name,column_name,row_count,null_count,missing_pct
0,btcusdt_15m_processed,candle_pattern_complex,303071,303036,99.9885
1,btcusdt_15m_processed,candle_pattern_3,303071,302923,99.9512
2,btcusdt_15m_processed,candle_pattern_1,303071,299697,98.8867
3,btcusdt_15m_processed,candle_pattern_2,303071,250121,82.5288
4,btcusdt_15m_processed,chart_pattern_confidence,303071,3344,1.1034
...,...,...,...,...,...
337,btcusdt_4h_processed,obv,18942,0,0.0000
338,btcusdt_4h_processed,ad_line,18942,0,0.0000
339,btcusdt_4h_processed,volume_oscillator,18942,0,0.0000
340,btcusdt_4h_processed,support,18942,0,0.0000


In [8]:
# Per-table missing data summary and top missing columns.
table_missing_summary = (
    missing_df.groupby("table_name", dropna=False)
    .agg(
        columns_with_missing=("null_count", lambda s: int((s > 0).sum())),
        avg_missing_pct=("missing_pct", "mean"),
        max_missing_pct=("missing_pct", "max"),
    )
    .reset_index()
    .sort_values(["max_missing_pct", "avg_missing_pct"], ascending=False)
)

top_missing_cols = missing_df[missing_df["null_count"] > 0].copy()
top_missing_cols = top_missing_cols.sort_values(["missing_pct", "null_count"], ascending=False).head(50)

print("Table-level missing summary:")
display(table_missing_summary)

print("Top 50 columns by missing percentage:")
display(top_missing_cols)

Table-level missing summary:


,table_name,columns_with_missing,avg_missing_pct,max_missing_pct
3,btcusdt_1mo_processed,15,6.967423,100.0000
1,btcusdt_1d_processed,15,6.715895,100.0000
4,btcusdt_1w_processed,15,6.466386,100.0000
2,btcusdt_1h_processed,29,7.038393,99.9947
5,btcusdt_4h_processed,23,7.005139,99.9947
0,btcusdt_15m_processed,29,6.820812,99.9885


Top 50 columns by missing percentage:


,table_name,column_name,row_count,null_count,missing_pct
57,btcusdt_1d_processed,candle_pattern_complex,3158,3158,100.0000
228,btcusdt_1w_processed,candle_pattern_complex,452,452,100.0000
171,btcusdt_1mo_processed,candle_pattern_complex,105,105,100.0000
114,btcusdt_1h_processed,candle_pattern_complex,75768,75764,99.9947
285,btcusdt_4h_processed,candle_pattern_complex,18942,18941,99.9947
0,btcusdt_15m_processed,candle_pattern_complex,303071,303036,99.9885
115,btcusdt_1h_processed,candle_pattern_3,75768,75734,99.9551
1,btcusdt_15m_processed,candle_pattern_3,303071,302923,99.9512
286,btcusdt_4h_processed,candle_pattern_3,18942,18928,99.9261
58,btcusdt_1d_processed,candle_pattern_3,3158,3146,99.6200
